# Data cleaning walkthrough on the **Titanic dataset**.


1. Load data
2. Detect problems
3. Apply **ALL relevant data cleaning techniques**
4. Validate the cleaned result


---

# Load Titanic Bad Dataset

In [325]:
import pandas as pd
import numpy as np

df = pd.read_csv("bad_data_titanic.csv")
df.head()

,passengerId,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


| Column          | Meaning                                                              |
| --------------- | -------------------------------------------------------------------- |
| **PassengerId** | Unique ID assigned to each passenger                                 |
| **Survived**    | Survival outcome (0 = No, 1 = Yes)                                   |
| **Pclass**      | Passenger class (1 = 1st, 2 = 2nd, 3 = 3rd)                          |
| **Name**        | Full name of the passenger (often includes title)                    |
| **Sex**         | Gender of the passenger                                              |
| **Age**         | Age in years (some values missing)                                   |
| **SibSp**       | Number of siblings or spouses aboard                                 |
| **Parch**       | Number of parents or children aboard                                 |
| **Ticket**      | Ticket number                                                        |
| **Fare**        | Amount paid for the ticket                                           |
| **Cabin**       | Cabin number (many missing values)                                   |
| **Embarked**    | Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton) |


# Initial Data Inspection (Always First)

In [326]:
print(df.shape) # rows, col

(901, 12)


In [327]:
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 901 entries, 0 to 900
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   passengerId  901 non-null    int64  
 1   survived     901 non-null    int64  
 2   pclass       901 non-null    int64  
 3   name         901 non-null    object 
 4   sex          901 non-null    object 
 5   age          709 non-null    float64
 6   sibsp        901 non-null    int64  
 7   parch        901 non-null    int64  
 8   ticket       901 non-null    object 
 9   fare         895 non-null    float64
 10  cabin        207 non-null    object 
 11  embarked     899 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 84.6+ KB


In [328]:
# 1) How many missing values ?
missing = df.isna().sum()
print(missing)

passengerId      0
survived         0
pclass           0
name             0
sex              0
age            192
sibsp            0
parch            0
ticket           0
fare             6
cabin          694
embarked         2
dtype: int64


In [329]:
# 2) show me percent

missing = df.isna().sum()
missing_percent = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Missing_Percentage': missing_percent.round(2)
})

print(missing_df)

             Missing_Count  Missing_Percentage
passengerId              0                0.00
survived                 0                0.00
pclass                   0                0.00
name                     0                0.00
sex                      0                0.00
age                    192               21.31
sibsp                    0                0.00
parch                    0                0.00
ticket                   0                0.00
fare                     6                0.67
cabin                  694               77.03
embarked                 2                0.22


In [330]:
# descriptive stats

print(df.describe())

       passengerId    survived      pclass         age       sibsp  \
count   901.000000  901.000000  901.000000  709.000000  901.000000   
mean    441.110988    0.385128    2.308546   29.949464    0.524972   
std     260.052428    0.486896    0.836809   15.738257    1.100852   
min       1.000000    0.000000    1.000000    0.420000    0.000000   
25%     216.000000    0.000000    2.000000   21.000000    0.000000   
50%     441.000000    0.000000    3.000000   28.000000    0.000000   
75%     666.000000    1.000000    3.000000   38.000000    1.000000   
max     891.000000    1.000000    3.000000  200.000000    8.000000   

            parch         fare  
count  901.000000   895.000000  
mean     0.380688    33.297704  
std      0.804449    59.241026  
min      0.000000     0.000000  
25%      0.000000     7.910400  
50%      0.000000    14.454200  
75%      0.000000    31.137500  
max      6.000000  1000.000000  


In [331]:
# names of columns
print(df.columns)

Index(['passengerId', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp',
       'parch', 'ticket', 'fare', 'cabin', 'embarked'],
      dtype='object')


In [332]:
# Show me value_counts for columns that are categorical in nature

cat_columns = ['survived', 'pclass', 'sex', 'sibsp', 'parch', 'embarked']

for col in cat_columns:
    print(df[col].value_counts())
    print("++++\n")

survived
0    554
1    347
Name: count, dtype: int64
++++

pclass
3    497
1    219
2    185
Name: count, dtype: int64
++++

sex
male      572
female    309
Female     10
MALE       10
Name: count, dtype: int64
++++

sibsp
0    613
1    213
2     28
4     18
3     17
8      7
5      5
Name: count, dtype: int64
++++

parch
0    686
1    119
2     81
5      5
3      5
4      4
6      1
Name: count, dtype: int64
++++

embarked
S    651
C    170
Q     78
Name: count, dtype: int64
++++



**Key points**

* Missing values- age, fare, cabin
* Wrong dtypes - survived should not be int64, etc
* Categorical imbalance - More passengers from Pclass=3
* Redundant features - passengerID, name 


---
# Detect Data Quality Issues

In [333]:
### Missing values

missing = df.isna().sum()

print(missing)

passengerId      0
survived         0
pclass           0
name             0
sex              0
age            192
sibsp            0
parch            0
ticket           0
fare             6
cabin          694
embarked         2
dtype: int64


In [334]:
### Missing values

missing = df.isna().sum()
missing_percent = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Missing_Percentage': missing_percent.round(2)
})

print(missing_df)

             Missing_Count  Missing_Percentage
passengerId              0                0.00
survived                 0                0.00
pclass                   0                0.00
name                     0                0.00
sex                      0                0.00
age                    192               21.31
sibsp                    0                0.00
parch                    0                0.00
ticket                   0                0.00
fare                     6                0.67
cabin                  694               77.03
embarked                 2                0.22


In [335]:
### Duplicate rows
print(df.duplicated().sum())

9


In [336]:
### Invalid values: Lets look at age

print(df["age"].describe())

# age ranges from 0.42 to 200. There are some invalid values here
# Minimum: 0 (infant) --> fine
# Maximum: ~80 (maybe 90)

# Age = 200 is either:
# data entry error
# OCR error
# placeholder value

count    709.000000
mean      29.949464
std       15.738257
min        0.420000
25%       21.000000
50%       28.000000
75%       38.000000
max      200.000000
Name: age, dtype: float64


In [337]:
### Category inconsistencies (for sex)

print(df["sex"].value_counts())


sex
male      572
female    309
Female     10
MALE       10
Name: count, dtype: int64


---
# DATA CLEANING (CORE SECTION)
---

In [338]:
## Remove Duplicates

df = df.drop_duplicates()

In [339]:
## Fix Categorical Inconsistencies
df = df.copy()

df["sex"] = (
    df["sex"]
    .str.lower()
    .str.strip()
)


print(df["sex"].value_counts())

sex
male      578
female    314
Name: count, dtype: int64


---

##  Handle Missing Values (Multiple Strategies)

In [340]:
### AGE → Median (robust to outliers)
median = df["age"].median()

df["age"] = df["age"].fillna(median)


In [341]:
### FARE → Median

median = df["fare"].median()
df["fare"] = df["fare"].fillna(median)

In [342]:
### EMBARK_TOWN → Mode

df["embarked"] = df["embarked"].fillna(
    df["embarked"].mode()[0]
)


In [343]:
### DROP columns with too many missing values: deck was missing >50% percent

df = df.drop(columns=["cabin"])

---

## Fix Invalid / Impossible Values


In [344]:
### Age > 100 is unrealistic

median = df["age"].median()
df.loc[df["age"] > 100, "age"] = median 


In [345]:
### Fare too high: Fill high fare > 500 with median

median = df["fare"].median()
df.loc[df["fare"] > 500, "fare"] = median

---

## Data Type Optimization (VERY IMPORTANT)

In [346]:
# step1: Before- Display data types and memory usage

print("Before memory usage:", df.info(memory_usage="deep"))

<class 'pandas.core.frame.DataFrame'>
Index: 892 entries, 0 to 891
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   passengerId  892 non-null    int64  
 1   survived     892 non-null    int64  
 2   pclass       892 non-null    int64  
 3   name         892 non-null    object 
 4   sex          892 non-null    object 
 5   age          892 non-null    float64
 6   sibsp        892 non-null    int64  
 7   parch        892 non-null    int64  
 8   ticket       892 non-null    object 
 9   fare         892 non-null    float64
 10  embarked     892 non-null    object 
dtypes: float64(2), int64(5), object(4)
memory usage: 288.7 KB
Before memory usage: None


In [347]:
# step2
df["pclass"] = df["pclass"].astype("int8")
df["survived"] = df["survived"].astype("int8")
df["age"] = df["age"].astype("int8")
df["sibsp"] = df["sibsp"].astype("int8")
df["parch"] = df["parch"].astype("int8")
df["fare"] = df["fare"].astype("float32")


In [348]:
# Convert categoricals:

cat_cols = [
    "sex", "embarked"
]

df[cat_cols] = df[cat_cols].astype("category")


In [349]:
print(df.sample(10))

     passengerId  survived  pclass                              name     sex  \
364          365         0       3               O'Brien, Mr. Thomas    male   
840          841         0       3       Alhomaki, Mr. Ilmari Rudolf    male   
613          614         0       3                  Horgan, Mr. John    male   
57            58         0       3               Novel, Mr. Mansouer    male   
13            14         0       3       Andersson, Mr. Anders Johan    male   
15            16         1       2  Hewlett, Mrs. (Mary D Kingcome)   female   
695          696         0       2        Chapman, Mr. Charles Henry    male   
120          121         0       2       Hickman, Mr. Stanley George    male   
435          436         1       1         Carter, Miss. Lucile Polk  female   
122          123         0       2              Nasser, Mr. Nicholas    male   

     age  sibsp  parch            ticket        fare embarked  
364   28      1      0            370365   15.500000   

In [350]:
# show memory usage after
print("After memory usage:", df.info(memory_usage="deep"))

<class 'pandas.core.frame.DataFrame'>
Index: 892 entries, 0 to 891
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   passengerId  892 non-null    int64   
 1   survived     892 non-null    int8    
 2   pclass       892 non-null    int8    
 3   name         892 non-null    object  
 4   sex          892 non-null    category
 5   age          892 non-null    int8    
 6   sibsp        892 non-null    int8    
 7   parch        892 non-null    int8    
 8   ticket       892 non-null    object  
 9   fare         892 non-null    float32 
 10  embarked     892 non-null    category
dtypes: category(2), float32(1), int64(1), int8(5), object(2)
memory usage: 152.7 KB
After memory usage: None


In [351]:
# percentage reductions by choosing right data type: 47%%
(288 - 152)/288

0.4722222222222222

**Huge memory reduction**


## Outlier Detection & Treatment (IQR Method)

In [353]:
# 1) find outliers
Q1 = df["fare"].quantile(0.25)
Q3 = df["fare"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers_df = df[(df["fare"] < lower) | (df["fare"] > upper)]
print(outliers_df)
print("Number of outliers:", len(outliers_df))

     passengerId  survived  pclass  \
1              2         1       1   
27            28         0       1   
31            32         1       1   
34            35         0       1   
52            53         1       1   
..           ...       ...     ...   
846          847         0       3   
849          850         1       1   
856          857         1       1   
863          864         0       3   
879          880         1       1   

                                                  name     sex  age  sibsp  \
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female   38      1   
27                      Fortune, Mr. Charles Alexander    male   19      3   
31      Spencer, Mrs. William Augustus (Marie Eugenie)  female   28      1   
34                             Meyer, Mr. Edgar Joseph    male   28      1   
52            Harper, Mrs. Henry Sleeper (Myna Haxtun)  female   49      1   
..                                                 ...     ...  ...    ... 

In [354]:
# 2)  cap them
df["fare"] = np.clip(df["fare"], lower, upper)

outliers_df = df[(df["fare"] < lower) | (df["fare"] > upper)]
print(outliers_df)
print("Number of outliers:", len(outliers_df))

Empty DataFrame
Columns: [passengerId, survived, pclass, name, sex, age, sibsp, parch, ticket, fare, embarked]
Index: []
Number of outliers: 0


## Feature Engineering (Cleaning Adjacent)


In [309]:
# Create new clean feature:

df["family_size"] = df["sibsp"] + df["parch"] + 1

# Bin age:

df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 12, 18, 60, 100],
    labels=["Child", "Teen", "Adult", "Senior"],
    include_lowest=True
)

print(df.sample(10))

     passengerId  survived  pclass                           name     sex  \
777          778         1       3  Emanuel, Miss. Virginia Ethel  female   
467          468         0       1     Smart, Mr. John Montgomery    male   
57            58         0       3            Novel, Mr. Mansouer    male   
702          703         0       3          Barbara, Miss. Saiide  female   
294          295         0       3               Mineff, Mr. Ivan    male   
645          646         1       1      Harper, Mr. Henry Sleeper    male   
4              5         0       3       Allen, Mr. William Henry    male   
203          204         0       3           Youseff, Mr. Gerious    male   
568          569         0       3            Doharr, Mr. Tannous    male   
138          139         0       3            Osen, Mr. Olaf Elon    male   

     age  sibsp  parch    ticket       fare embarked  family_size age_group  
777    5      0      0    364516  12.475000        S            1     Chil

In [310]:
# 1) Extract title from names

df['title'] = df['name'].str.extract(r', (\w+)\.')  # extract text between ',' and '.'
print(df['title'].value_counts())

title
Mr          518
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Major         2
Mlle          2
Col           2
Don           1
Mme           1
Ms            1
Lady          1
Sir           1
Capt          1
Jonkheer      1
Name: count, dtype: int64

In [311]:
# 2) Create rare title

rare_titles = ['Rev', 'Dr', 'Major', 'Col', 'Sir', 'Lady', 'Countess', 'Capt', 'Don', 'Jonkheer']
df['title'] = df['title'].replace(rare_titles, 'RareTitle')


In [312]:
# 3) Some anomalies in Sex can be cross-checked using Title: No anamolies
df[df['title'].isin(['Master', 'Mr']) & (df['sex'] != 'male')]


,passengerId,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,embarked,family_size,age_group,title


In [313]:
# Lets Look at surnames

df['Surname'] = df['name'].str.split(',').str[0]
print(df['Surname'].value_counts())


Surname
Andersson    9
Sage         7
Skoog        6
Panula       6
Johnson      6
            ..
Hanna        1
Lewy         1
Mineff       1
Haas         1
Dooley       1
Name: count, Length: 667, dtype: int64

In [314]:
# Lets create column family size

df['familysize'] = df['sibsp'] + df['parch'] + 1
df['is_alone'] = 1  # default
df.loc[df['familysize'] > 1, 'is_alone'] = 0

In [356]:
# Drop columns that is not needed:
df = df.drop(columns=["passengerId", "name", "ticket"])

## Validate Cleaned Dataset

In [357]:
print(df.info())

# Here if age column is not needed, then it can be removed

<class 'pandas.core.frame.DataFrame'>
Index: 892 entries, 0 to 891
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   survived  892 non-null    int8    
 1   pclass    892 non-null    int8    
 2   sex       892 non-null    category
 3   age       892 non-null    int8    
 4   sibsp     892 non-null    int8    
 5   parch     892 non-null    int8    
 6   fare      892 non-null    float64 
 7   embarked  892 non-null    category
dtypes: category(2), float64(1), int8(5)
memory usage: 20.3 KB
None


In [316]:
# 1) missing values

print(df.isna().sum())

passengerId    0
survived       0
pclass         0
name           0
sex            0
age            0
sibsp          0
parch          0
ticket         0
fare           0
embarked       0
family_size    0
age_group      0
title          1
Surname        0
familysize     0
is_alone       0
dtype: int64


In [317]:
# 2) Show me rows that are missing values

missing_rows = df[df.isna().any(axis=1)]
print(missing_rows)
print(missing_rows.to_dict(orient="records"))

     passengerId  survived  pclass  \
759          760         1       1   

                                                  name     sex  age  sibsp  \
759  Rothes, the Countess. of (Lucy Noel Martha Dye...  female   33      0   

     parch  ticket     fare embarked  family_size age_group title Surname  \
759      0  110152  64.4063        S            1     Adult   NaN  Rothes   

     familysize  is_alone  
759           1         1  
[{'passengerId': 760, 'survived': 1, 'pclass': 1, 'name': 'Rothes, the Countess. of (Lucy Noel Martha Dyer-Edwards)', 'sex': 'female', 'age': 33, 'sibsp': 0, 'parch': 0, 'ticket': '110152', 'fare': 64.40629982948303, 'embarked': 'S', 'family_size': 1, 'age_group': 'Adult', 'title': nan, 'Surname': 'Rothes', 'familysize': 1, 'is_alone': 1}]


In [318]:
# 3) manually change title

df.loc[759, 'title'] = 'RareTitle'
print(df.isna().sum())

passengerId    0
survived       0
pclass         0
name           0
sex            0
age            0
sibsp          0
parch          0
ticket         0
fare           0
embarked       0
family_size    0
age_group      0
title          0
Surname        0
familysize     0
is_alone       0
dtype: int64


In [319]:
# descriptive stats
print(df.describe())

       passengerId    survived      pclass         age       sibsp  \
count   892.000000  892.000000  892.000000  892.000000  892.000000   
mean    445.501121    0.383408    2.309417   29.316143    0.523543   
std     257.640579    0.486489    0.835923   12.839720    1.102240   
min       1.000000    0.000000    1.000000    0.000000    0.000000   
25%     222.750000    0.000000    2.000000   22.000000    0.000000   
50%     445.500000    0.000000    3.000000   28.000000    0.000000   
75%     668.250000    1.000000    3.000000   35.000000    1.000000   
max     891.000000    1.000000    3.000000   80.000000    8.000000   

            parch        fare  family_size  familysize    is_alone  
count  892.000000  892.000000   892.000000  892.000000  892.000000  
mean     0.381166   23.629960     1.904709    1.904709    0.602018  
std      0.805706   20.021101     1.612556    1.612556    0.489756  
min      0.000000    0.000000     1.000000    1.000000    0.000000  
25%      0.000000    7.8

In [320]:
# value count 
print(df["sex"].value_counts())

sex
male      578
female    314
Name: count, dtype: int64


In [358]:
print(df.sample(10))

     survived  pclass     sex  age  sibsp  parch       fare embarked
364         0       3    male   28      1      0  15.500000        Q
733         0       2    male   23      0      0  13.000000        S
334         1       1  female   28      1      0  64.406300        S
859         0       3    male   28      0      0   7.229200        C
285         0       3    male   33      0      0   8.662500        C
576         1       2  female   34      0      0  13.000000        S
512         1       1    male   36      0      0  26.287500        S
571         1       1  female   53      2      0  51.479198        S
131         0       3    male   20      0      0   7.050000        S
156         1       3  female   16      0      0   7.733300        Q


---

#  BEFORE vs AFTER 

| Issue          | Before  | After     |
| -------------- | ------- | --------- |
| Missing values | Many    | 0         |
| Duplicates     | Present | Removed   |
| Invalid ages   | Yes     | Fixed     |
| Category noise | Yes     | Clean     |
| Memory usage   | High    | Optimized |
| ML readiness   | NO      | YES       |
